<a href="https://colab.research.google.com/github/proinvestigadores/Abejas/blob/main/Smart_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Carga y auditoría de datos**

## **1.1. Setup y carga de datos**

In [6]:
# ---- 0) Configuración básica de entorno
import sys, os, re, math, json, numpy as np, pandas as pd
from datetime import datetime, timedelta

# Ajusta pandas para ver más columnas en prints
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

# ---- 1) Montar Google Drive y definir ruta del CSV
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Importar
folder_path = '/content/drive/MyDrive/Pigs/Smart_Manufacturing/Modelos/'
file_path = 'Bd_Procesada_Junio_2024.csv'
path = folder_path + file_path
df = pd.read_csv(path)
df.head()

Mounted at /content/drive


,Timestamp,MD1_RPM,MD2_RPM,MD3_RPM,MD4_RPM,MD5_RPM,MD1_P,MD2_P,MD3_P,MD4_P,MD5_P,MD1_APM,MD2_APM,MD3_APM,MD4_APM,MD5_APM,MD1_PORCENT,MD2_PORCENT,MD3_PORCENT,MD4_PORCENT,MD5_PORCENT,MD1_ZE_VP,MD2_ZE_VP,MD3_ZE_VP,MD4_ZE_VP,MD5_ZE_VP,MD1_FND,MD2_FND,MD3_FND,MD4_FND,MD5_FND,CBZ_Z1_SP,CBZ_Z1_VP,CBZ_Z2_SP,CBZ_Z2_VP,CBZ_Z3_SP,CBZ_Z3_VP,CBZ_Z4_SP,CBZ_Z4_VP,CBZ_Z5_SP,CBZ_Z5_VP,CBZ_Z6_SP,CBZ_Z6_VP,HAL1_VEL,HAL2_VEL,HAL3_VEL,HAL4_VEL,HAL5_VEL,HAL6_VEL
0,2024-03-19 16:30:11,20.0,3.0,3.0,3.0,10.0,4678.0,1322.0,245.0,805.0,945.0,12.100,12.495,1.7,1.7,18.380,0.88,1.32,1.32,1.39,1.23,35.8,31.5,34.9,34.0,34.2,273.1,258.1,269.6,263.9,270.4,249.0,251.4,249.0,249.0,250.0,249.7,250.0,253.6,251.0,250.8,251.0,251.0,40.4,42.2,34.0,94.5,95.3,73.0
1,2024-03-19 16:30:59,20.0,3.0,3.0,3.0,10.0,4678.0,1322.0,245.0,805.0,945.0,12.100,12.495,1.7,1.7,18.380,0.88,1.32,1.32,1.39,1.23,35.8,31.5,34.9,34.0,34.2,273.1,258.1,269.6,263.9,270.4,249.0,251.4,249.0,249.0,250.0,249.7,250.0,253.6,251.0,250.8,251.0,251.0,40.4,42.2,34.0,94.5,95.3,73.0
2,2024-03-19 16:31:58,10.0,3.0,3.0,3.0,10.0,3880.0,1250.0,225.0,795.0,928.0,11.570,12.530,1.7,1.7,18.365,0.88,1.32,1.32,1.39,1.23,35.2,31.5,34.6,34.0,34.6,273.9,255.0,271.8,264.5,269.4,249.0,251.9,249.0,249.0,250.0,249.7,250.0,253.8,251.0,250.8,251.0,250.9,40.4,42.2,34.0,94.5,95.3,73.0
3,2024-03-19 16:32:58,10.0,3.0,3.0,3.0,5.0,2928.0,1160.0,180.0,762.0,728.0,10.985,12.485,1.7,1.7,17.300,0.88,1.32,1.32,1.39,1.23,35.7,31.5,34.4,34.0,34.7,276.6,251.9,270.0,267.5,268.4,249.0,252.3,249.0,249.0,250.0,249.8,250.0,253.7,251.0,250.8,251.0,250.9,40.4,42.2,34.0,94.5,95.3,73.0
4,2024-03-19 16:33:58,10.0,3.0,3.0,3.0,5.0,2770.0,1137.0,158.0,735.0,548.0,10.960,12.420,1.7,1.7,16.125,0.88,1.32,1.32,1.39,1.23,35.8,31.4,34.4,34.0,34.2,279.4,249.0,266.8,270.5,267.5,249.0,252.4,249.0,249.2,250.0,250.0,250.0,253.4,251.0,250.8,251.0,250.9,40.4,42.2,34.0,94.5,95.3,73.0


# **1.2. Funciones de auditoría de datos**

In [7]:
# =========================================
# BLOQUE 1 — SETUP + CARGA + AUDITORÍA
# Objetivo 3: Validación del modelo (paso 1)
# =========================================

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Estandariza nombres de columnas:
    - elimina espacios al inicio/fin
    - colapsa múltiples espacios internos
    - reemplaza espacios por '_'
    - asegura ASCII simple si hubiera tildes
    """
    def fix_col(c):
        # quita espacios externos
        c = c.strip()
        # colapsa espacios intermedios
        c = re.sub(r'\s+', ' ', c)
        # reemplaza espacios por underscore
        c = c.replace(' ', '_')
        # normaliza prefijos comunes tipo 'MD1' etc. sin tocar mayúsculas útiles
        return c
    df = df.rename(columns={c: fix_col(c) for c in df.columns})
    return df


def parse_timestamp(df: pd.DataFrame, ts_col_candidates=('Timestamp','timestamp','TIMESTAMP')) -> pd.DataFrame:
    """
    Busca la columna de tiempo, la parsea a datetime y ordena por tiempo.
    """
    ts_col = None
    for c in ts_col_candidates:
        if c in df.columns:
            ts_col = c
            break
    if ts_col is None:
        raise ValueError(f"No encontré columna de tiempo. Probé {ts_col_candidates} y no existen en df.columns={df.columns.tolist()[:10]}...")

    # Parse robusto (maneja 'YYYY-MM-DD HH:MM:SS' y variantes)
    df[ts_col] = pd.to_datetime(df[ts_col], errors='coerce')
    if df[ts_col].isna().any():
        # Si hay fallos de parseo, explícitalos
        bad = df[df[ts_col].isna()]
        print("⚠️ Filas con Timestamp inválido:", len(bad))
        print(bad.head(3))
        # Puedes decidir aquí si las eliminas
        df = df.dropna(subset=[ts_col])

    df = df.sort_values(ts_col).reset_index(drop=True)
    return df, ts_col


def build_variable_families(columns):
    """
    Detecta familias de variables según prefijos/sufijos habituales del proceso:
    - RPM de módulos: MD*_RPM
    - Presiones: MD*_P (PSI)
    - % capa: MD*_PORCENT
    - Temperatura de fundido: MD*_FND
    - Temperaturas de cabezal: CBZ_Z{1..6}_SP y CBZ_Z{1..6}_VP
    - Velocidad haladores: HAL{1..6}_VEL
    Devuelve un dict con listas de columnas por familia.
    """
    fam = {
        "rpm_modulos": sorted([c for c in columns if re.fullmatch(r"MD\d+_RPM", c)]),
        "presion_modulos": sorted([c for c in columns if re.fullmatch(r"MD\d+_P", c)]),
        "porcentaje_capa": sorted([c for c in columns if re.fullmatch(r"MD\d+_PORCENT", c)]),
        "fundido_modulos": sorted([c for c in columns if re.fullmatch(r"MD\d+_FND", c)]),
        "cabezal_SP": sorted([c for c in columns if re.fullmatch(r"CBZ_Z\d+_SP", c)]),
        "cabezal_VP": sorted([c for c in columns if re.fullmatch(r"CBZ_Z\d+_VP", c)]),
        "haladores_vel": sorted([c for c in columns if re.fullmatch(r"HAL\d+_VEL", c)]),
    }
    # Extras comunes en tu BD (si existen):
    extras = sorted([c for c in columns if c not in sum(fam.values(), []) and c not in ("Timestamp","timestamp","TIMESTAMP")])
    fam["extras"] = extras
    return fam


def time_gap_audit(ts: pd.Series) -> pd.DataFrame:
    """
    Calcula deltas consecutivos y los clasifica en bins relevantes para este proceso:
     - < 30s
     - 30s–59s
     - 59s–61s (ideal ≈ 60s)
     - 61s–120s
     - 2–5 min
     - 5–10 min
     - >10 min
    Devuelve tabla con conteos y %.
    """
    # diferencias en segundos
    dt = ts.sort_values().diff().dt.total_seconds().iloc[1:]  # omite la primera (NaN)
    bins = [-np.inf, 30, 59, 61, 120, 300, 600, np.inf]
    labels = ["<30s", "30–59s", "59–61s", "61–120s", "2–5min", "5–10min", ">10min"]
    cat = pd.cut(dt, bins=bins, labels=labels)
    counts = cat.value_counts().reindex(labels, fill_value=0)
    out = pd.DataFrame({
        "rango": counts.index,
        "conteo": counts.values,
        "porcentaje": np.round(100*counts.values / counts.sum(), 2)
    })
    out.loc["TOTAL"] = ["—", int(counts.sum()), 100.00]
    return out

# Carga del CSV
assert os.path.exists(path), f"No encuentro el CSV en: {path}"
df = pd.read_csv(path)

print("Forma (filas, columnas) antes de estandarizar:", df.shape)
df = standardize_columns(df)
df, TS_COL = parse_timestamp(df)

print("Columna de tiempo detectada:", TS_COL)
print("Rango temporal:", df[TS_COL].min(), "→", df[TS_COL].max())
print("Forma (filas, columnas) tras parseo/orden:", df.shape)
print("\nPrimeras filas:")
display(df.head(5))

# Diccionario de variables por familia
familias = build_variable_families(df.columns)
print("\nResumen de familias detectadas:")
for k, v in familias.items():
    print(f" - {k}: {len(v)} columnas")
    if len(v) > 0:
        print("   ", v)

# Auditoría de gaps temporales
gap_table = time_gap_audit(df[TS_COL])
print("\n=== Auditoría de gaps entre muestras ===")
display(gap_table)

# Chequeos rápidos de calidad de datos
# a) Porcentaje de valores nulos por columna (útil para decidir interpolaciones posteriores)
nulls = df.isna().mean().sort_values(ascending=False)
print("\nTop 10 columnas con mayor % de NaN:")
display((nulls*100).round(2).head(10).to_frame("%_NaN").T)

# b) Duplicados exactos por Timestamp (según milisegundo).
#    En el proceso, lo ideal es consolidar a muestreo por minuto en un bloque posterior.
dup_count = df.duplicated(subset=[TS_COL]).sum()
print(f"\nDuplicados exactos de {TS_COL}: {dup_count}")

# c) Vista de estadísticos básicos para familias principales (sirve para detectar sensores fuera de rango):
def describe_family(cols, titulo):
    if len(cols) == 0:
        print(f"\n[{titulo}] No se encontraron columnas.")
        return
    print(f"\n[{titulo}] Estadísticos rápidos")
    display(df[cols].describe().T)

describe_family(familias["rpm_modulos"], "RPM de módulos")
describe_family(familias["presion_modulos"], "Presión de módulos (PSI)")
describe_family(familias["fundido_modulos"], "Temperatura de fundido (°C)")
describe_family(familias["cabezal_SP"], "Cabezal SP (°C)")
describe_family(familias["cabezal_VP"], "Cabezal VP (°C)")
describe_family(familias["haladores_vel"], "Velocidad haladores (m/min aprox.)")

print("\n✅ BLOQUE 1 completado. Sube estas salidas al repo para continuar con el Bloque 2.")


Forma (filas, columnas) antes de estandarizar: (20260, 49)
Columna de tiempo detectada: Timestamp
Rango temporal: 2024-03-19 16:30:11 → 2024-04-02 16:50:06
Forma (filas, columnas) tras parseo/orden: (20260, 49)

Primeras filas:


,Timestamp,MD1_RPM,MD2_RPM,MD3_RPM,MD4_RPM,MD5_RPM,MD1_P,MD2_P,MD3_P,MD4_P,MD5_P,MD1_APM,MD2_APM,MD3_APM,MD4_APM,MD5_APM,MD1_PORCENT,MD2_PORCENT,MD3_PORCENT,MD4_PORCENT,MD5_PORCENT,MD1_ZE_VP,MD2_ZE_VP,MD3_ZE_VP,MD4_ZE_VP,MD5_ZE_VP,MD1_FND,MD2_FND,MD3_FND,MD4_FND,MD5_FND,CBZ_Z1_SP,CBZ_Z1_VP,CBZ_Z2_SP,CBZ_Z2_VP,CBZ_Z3_SP,CBZ_Z3_VP,CBZ_Z4_SP,CBZ_Z4_VP,CBZ_Z5_SP,CBZ_Z5_VP,CBZ_Z6_SP,CBZ_Z6_VP,HAL1_VEL,HAL2_VEL,HAL3_VEL,HAL4_VEL,HAL5_VEL,HAL6_VEL
0,2024-03-19 16:30:11,20.0,3.0,3.0,3.0,10.0,4678.0,1322.0,245.0,805.0,945.0,12.100,12.495,1.7,1.7,18.380,0.88,1.32,1.32,1.39,1.23,35.8,31.5,34.9,34.0,34.2,273.1,258.1,269.6,263.9,270.4,249.0,251.4,249.0,249.0,250.0,249.7,250.0,253.6,251.0,250.8,251.0,251.0,40.4,42.2,34.0,94.5,95.3,73.0
1,2024-03-19 16:30:59,20.0,3.0,3.0,3.0,10.0,4678.0,1322.0,245.0,805.0,945.0,12.100,12.495,1.7,1.7,18.380,0.88,1.32,1.32,1.39,1.23,35.8,31.5,34.9,34.0,34.2,273.1,258.1,269.6,263.9,270.4,249.0,251.4,249.0,249.0,250.0,249.7,250.0,253.6,251.0,250.8,251.0,251.0,40.4,42.2,34.0,94.5,95.3,73.0
2,2024-03-19 16:31:58,10.0,3.0,3.0,3.0,10.0,3880.0,1250.0,225.0,795.0,928.0,11.570,12.530,1.7,1.7,18.365,0.88,1.32,1.32,1.39,1.23,35.2,31.5,34.6,34.0,34.6,273.9,255.0,271.8,264.5,269.4,249.0,251.9,249.0,249.0,250.0,249.7,250.0,253.8,251.0,250.8,251.0,250.9,40.4,42.2,34.0,94.5,95.3,73.0
3,2024-03-19 16:32:58,10.0,3.0,3.0,3.0,5.0,2928.0,1160.0,180.0,762.0,728.0,10.985,12.485,1.7,1.7,17.300,0.88,1.32,1.32,1.39,1.23,35.7,31.5,34.4,34.0,34.7,276.6,251.9,270.0,267.5,268.4,249.0,252.3,249.0,249.0,250.0,249.8,250.0,253.7,251.0,250.8,251.0,250.9,40.4,42.2,34.0,94.5,95.3,73.0
4,2024-03-19 16:33:58,10.0,3.0,3.0,3.0,5.0,2770.0,1137.0,158.0,735.0,548.0,10.960,12.420,1.7,1.7,16.125,0.88,1.32,1.32,1.39,1.23,35.8,31.4,34.4,34.0,34.2,279.4,249.0,266.8,270.5,267.5,249.0,252.4,249.0,249.2,250.0,250.0,250.0,253.4,251.0,250.8,251.0,250.9,40.4,42.2,34.0,94.5,95.3,73.0



Resumen de familias detectadas:
 - rpm_modulos: 5 columnas
    ['MD1_RPM', 'MD2_RPM', 'MD3_RPM', 'MD4_RPM', 'MD5_RPM']
 - presion_modulos: 5 columnas
    ['MD1_P', 'MD2_P', 'MD3_P', 'MD4_P', 'MD5_P']
 - porcentaje_capa: 5 columnas
    ['MD1_PORCENT', 'MD2_PORCENT', 'MD3_PORCENT', 'MD4_PORCENT', 'MD5_PORCENT']
 - fundido_modulos: 5 columnas
    ['MD1_FND', 'MD2_FND', 'MD3_FND', 'MD4_FND', 'MD5_FND']
 - cabezal_SP: 6 columnas
    ['CBZ_Z1_SP', 'CBZ_Z2_SP', 'CBZ_Z3_SP', 'CBZ_Z4_SP', 'CBZ_Z5_SP', 'CBZ_Z6_SP']
 - cabezal_VP: 6 columnas
    ['CBZ_Z1_VP', 'CBZ_Z2_VP', 'CBZ_Z3_VP', 'CBZ_Z4_VP', 'CBZ_Z5_VP', 'CBZ_Z6_VP']
 - haladores_vel: 6 columnas
    ['HAL1_VEL', 'HAL2_VEL', 'HAL3_VEL', 'HAL4_VEL', 'HAL5_VEL', 'HAL6_VEL']
 - extras: 10 columnas
    ['MD1_APM', 'MD1_ZE_VP', 'MD2_APM', 'MD2_ZE_VP', 'MD3_APM', 'MD3_ZE_VP', 'MD4_APM', 'MD4_ZE_VP', 'MD5_APM', 'MD5_ZE_VP']

=== Auditoría de gaps entre muestras ===


,rango,conteo,porcentaje
0,<30s,221,1.09
1,30–59s,3238,15.98
2,59–61s,14279,70.48
3,61–120s,2512,12.40
4,2–5min,9,0.04
5,5–10min,0,0.00
6,>10min,0,0.00
TOTAL,—,20259,100.00



Top 10 columnas con mayor % de NaN:


...


[Cabezal VP (°C)] Estadísticos rápidos


,count,mean,std,min,25%,50%,75%,max
CBZ_Z1_VP,20260.0,251.367270,0.766144,249.6,250.8,251.3,252.0,252.8
CBZ_Z2_VP,20260.0,248.725030,0.484118,244.6,248.3,249.0,249.0,249.9
CBZ_Z3_VP,20260.0,247.860464,0.757689,246.1,247.3,247.9,248.4,250.5
CBZ_Z4_VP,20260.0,251.406703,0.360667,250.7,251.1,251.4,251.7,253.9
CBZ_Z5_VP,20260.0,248.011841,0.251156,245.1,248.0,248.0,248.0,251.4
CBZ_Z6_VP,20260.0,248.012315,0.294222,243.8,247.9,248.0,248.1,252.5



[Velocidad haladores (m/min aprox.)] Estadísticos rápidos


,count,mean,std,min,25%,50%,75%,max
HAL1_VEL,20260.0,40.004980,1.046879,35.2,39.8,40.1,40.8,41.1
HAL2_VEL,20260.0,41.762665,1.092682,36.5,41.6,41.9,42.5,42.7
HAL3_VEL,20260.0,34.000000,0.000000,34.0,34.0,34.0,34.0,34.0
HAL4_VEL,20260.0,89.457295,2.645648,77.4,89.1,89.7,91.0,94.5
HAL5_VEL,20260.0,94.284970,2.450095,83.2,94.0,94.1,96.0,97.0
HAL6_VEL,20260.0,73.277853,1.930675,64.5,73.0,73.5,74.7,75.2



✅ BLOQUE 1 completado. Sube estas salidas al repo para continuar con el Bloque 2.
